# 🦴 Notebook 01 — Data Exploration (EDA)
## Knee Osteoarthritis KL Grading Project
**MSt Healthcare Data Science — University of Cambridge**

---

### What this notebook does
This notebook explores the Kaggle Knee Osteoarthritis Dataset **before any model training**.
It answers five questions that directly inform every methodological decision in the report:

1. How many images exist per KL grade — and how imbalanced is the dataset?
2. What do the five KL grades actually look like visually?
3. Are the images consistent in size, quality, and appearance?
4. What are the pixel intensity characteristics across grades?
5. What class weights should we use to address imbalance?

**No GPU needed.** Run this entirely on CPU in Colab.

---
### Before running: change runtime
Runtime → Change runtime type → CPU (no GPU needed for this notebook)

## CELL 1 — Mount Google Drive
Every session in Colab starts fresh — files you create disappear when the session ends.
Mounting Google Drive means everything saves permanently to your Drive folder.

In [ ]:
# This imports the 'drive' module from Google's Colab tools
# It only exists inside Colab — you cannot run this line on your Mac
from google.colab import drive

# This command connects your Google Drive to Colab
# '/content/drive' is the folder path inside Colab where Drive will appear
# You will see a popup asking you to authorise — click Allow
drive.mount('/content/drive')

# This sets a variable called PROJECT that stores the path to your project folder
# We use this variable everywhere so if you rename the folder, you only change it here
# Make sure this folder exists in your Google Drive before running
PROJECT = '/content/drive/MyDrive/knee_oa_kl_grading'

# Print the path so you can confirm it loaded correctly
print(f'Project folder: {PROJECT}')

## CELL 2 — Import Libraries
These are the tools we need for this notebook.
All of them are pre-installed in Colab — no pip install needed here.

In [ ]:
# os: lets us navigate folders and files on the computer
# We use it to list folders, count files, and build file paths
import os

# pathlib: a cleaner way to work with file paths
# Path('/some/folder') is easier to read than '/some' + '/' + 'folder'
from pathlib import Path

# numpy: the core library for numerical calculations in Python
# We use it to calculate means, standard deviations, and pixel statistics
import numpy as np

# pandas: for creating and displaying tables of data
# We use it to build our class distribution summary table
import pandas as pd

# matplotlib: the main Python plotting library
# pyplot is the submodule that works like MATLAB plotting commands
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # for creating grids of subplots

# seaborn: makes matplotlib plots look better with less code
# We use it for the class distribution bar chart
import seaborn as sns

# PIL (Pillow): the standard Python library for opening and reading image files
# Image is the class we use to open .jpg and .png files
from PIL import Image

# warnings: lets us suppress unhelpful warning messages
import warnings
warnings.filterwarnings('ignore')  # hides warnings that clutter the output

# random: for selecting random samples of images
import random
random.seed(42)  # setting seed 42 means we get the same random sample every run

print('All libraries imported successfully')

## CELL 3 — Define Dataset Paths
We tell Python where to find the train, validation, and test folders.
This assumes you have already uploaded the Kaggle dataset to Google Drive.

In [ ]:
# Define the path to the raw data folder inside your project
# Path() turns a string into a proper file path object
DATA_DIR = Path(PROJECT) / 'data' / 'raw'

# The Kaggle dataset is pre-split into three folders
# We define each one as a separate variable
TRAIN_DIR = DATA_DIR / 'train'  # ~5,778 images — used to train the model
VAL_DIR   = DATA_DIR / 'val'    # ~826 images  — used to check performance during training
TEST_DIR  = DATA_DIR / 'test'   # ~1,656 images — used ONLY for final evaluation

# KL grades go from 0 (normal) to 4 (severe)
# We store them as strings because the folder names are '0', '1', '2', '3', '4'
KL_GRADES = ['0', '1', '2', '3', '4']

# These are the clinical names for each grade
# We use these as labels in charts so the plots are clinically meaningful
GRADE_NAMES = {
    '0': 'KL0 — Normal',
    '1': 'KL1 — Doubtful',
    '2': 'KL2 — Mild',
    '3': 'KL3 — Moderate',
    '4': 'KL4 — Severe'
}

# Check the folders actually exist before we go further
# If this prints False, your data is not in the right place
for split_name, split_path in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    exists = split_path.exists()  # .exists() returns True or False
    print(f'{split_name}: {split_path} — exists: {exists}')

## CELL 4 — Count Images Per Grade
This is the most important cell in the EDA.
It tells us how imbalanced the dataset is — which directly motivates
our use of class-weighted loss in the training experiments.

In [ ]:
# We will build a dictionary to store counts for each split
# A dictionary maps keys to values: {'train': {...}, 'val': {...}, 'test': {...}}
counts = {}

# Loop through each split (train, val, test)
# zip() pairs up the names and paths so we can loop through both at once
for split_name, split_path in zip(
    ['train', 'val', 'test'],
    [TRAIN_DIR, VAL_DIR, TEST_DIR]
):
    # For each split, create an inner dictionary for grade counts
    counts[split_name] = {}

    # Loop through each KL grade folder (0, 1, 2, 3, 4)
    for grade in KL_GRADES:
        grade_path = split_path / grade  # full path to this grade's folder

        if grade_path.exists():  # only count if the folder exists
            # os.listdir returns a list of all files in the folder
            # len() counts how many files are in that list
            # This gives us the number of images for this grade
            n_images = len(os.listdir(grade_path))
            counts[split_name][grade] = n_images
        else:
            counts[split_name][grade] = 0  # folder missing = 0 images

# Convert the counts dictionary to a pandas DataFrame
# A DataFrame is like an Excel table — rows and columns
# orient='index' means each split becomes a row
df_counts = pd.DataFrame(counts).T  # .T transposes rows and columns

# Rename columns to include clinical grade names
df_counts.columns = [GRADE_NAMES[g] for g in KL_GRADES]

# Add a 'Total' column that sums across all grades for each split
df_counts['Total'] = df_counts.sum(axis=1)  # axis=1 means sum across columns

print('Image counts per KL grade:')
print(df_counts)
print()

# Calculate the imbalance ratio for the training set
# This is the number you cite in your Methods section
train_counts = [counts['train'][g] for g in KL_GRADES]
max_count = max(train_counts)  # grade with most images
min_count = min(train_counts)  # grade with fewest images
imbalance_ratio = max_count / min_count

print(f'Maximum class count (train): {max_count}')
print(f'Minimum class count (train): {min_count}')
print(f'Imbalance ratio: {imbalance_ratio:.1f}:1')
print()
print('This imbalance ratio is what you cite in your Methods section.')
print('It justifies your use of class-weighted loss and WeightedRandomSampler.')

## CELL 5 — Class Distribution Bar Chart
Visualise the imbalance. This figure shows your marker exactly why
standard accuracy is a misleading metric for this dataset.

In [ ]:
# Create a figure with three side-by-side plots (one per split)
# figsize=(15, 5) sets the width to 15 inches and height to 5 inches
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# These colours correspond to disease severity
# Green for normal, yellow for doubtful, orange/red for disease
GRADE_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

# Loop through each split and plot a bar chart
for ax, (split_name, split_path) in zip(
    axes,
    zip(['Train', 'Validation', 'Test'], [TRAIN_DIR, VAL_DIR, TEST_DIR])
):
    # Get counts for this split
    grade_counts = [counts[split_name.lower()][g] for g in KL_GRADES]
    grade_labels = [f'KL{g}' for g in KL_GRADES]

    # Create the bar chart
    # ax.bar() draws bars: x positions, heights, and colours
    bars = ax.bar(grade_labels, grade_counts, color=GRADE_COLORS, 
                  edgecolor='white', linewidth=1.5)

    # Add the exact count on top of each bar so the reader can see the numbers
    for bar, count in zip(bars, grade_counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,  # x position: centre of bar
            bar.get_height() + 20,               # y position: just above bar
            str(count),                          # the number to display
            ha='center', va='bottom',            # centre-align the text
            fontsize=10, fontweight='bold'
        )

    # Labels and formatting
    ax.set_title(f'{split_name} Set (n={sum(grade_counts)})', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('KL Grade', fontsize=11)
    ax.set_ylabel('Number of Images', fontsize=11)
    ax.spines['top'].set_visible(False)    # remove top border
    ax.spines['right'].set_visible(False)  # remove right border

# Main title for the whole figure
fig.suptitle('Class Distribution Across KL Grades', 
             fontsize=15, fontweight='bold', y=1.02)

# Adjust spacing between plots so they don't overlap
plt.tight_layout()

# Save the figure to Drive so it persists after the session ends
# bbox_inches='tight' removes unnecessary white space around the figure
# dpi=150 sets resolution — 150 is good for reports, 300 for publication
save_path = f"{PROJECT}/outputs/figures/class_distribution.png"
os.makedirs(os.path.dirname(save_path), exist_ok=True)  # create folder if needed
plt.savefig(save_path, bbox_inches='tight', dpi=150)
print(f'Figure saved to: {save_path}')

plt.show()

## CELL 6 — Calculate Class Weights
These are the actual numbers you will plug into your training script.
Class weights tell the loss function to penalise errors on rare grades more heavily.

**Formula:** weight for grade i = total images / (number of grades × images in grade i)

This means rare grades get a higher weight — errors on KL1 are penalised more than errors on KL0.

In [ ]:
# Get the training counts as a numpy array for calculations
train_counts_array = np.array([counts['train'][g] for g in KL_GRADES], dtype=np.float32)

# Total number of training images
total_train = train_counts_array.sum()

# Number of classes (always 5 for KL grades 0-4)
n_classes = len(KL_GRADES)

# Calculate inverse frequency weights
# Grades with fewer images get a higher weight
# np.float32 is a specific number type that PyTorch expects
class_weights = total_train / (n_classes * train_counts_array)

# Normalise weights so they sum to the number of classes
# This keeps the overall loss on the same scale as unweighted loss
class_weights = class_weights / class_weights.sum() * n_classes

# Print the weights with clinical labels
print('Class weights for weighted CrossEntropyLoss:')
print('(Higher weight = model penalised more for errors on this grade)')
print()

for grade, name, weight, count in zip(
    KL_GRADES, 
    [GRADE_NAMES[g] for g in KL_GRADES],
    class_weights,
    train_counts_array
):
    print(f'  Grade {grade} ({name.split("—")[1].strip():12s}): '
          f'count={int(count):5d}  weight={weight:.4f}')

print()
print('Copy these weights into your train.py config:')
print(f'class_weights = torch.tensor({[round(float(w), 4) for w in class_weights]}, dtype=torch.float32)')

## CELL 7 — Example Images Per KL Grade
This generates **Figure 1** in your report.
It shows one example knee X-ray for each KL grade so the reader
can see what the model is trying to distinguish.

As a radiographer, look at these carefully:
- Can YOU tell the difference between KL0 and KL1 at a glance?
- The answer will inform your Discussion section.

In [ ]:
# Create a figure with 5 columns (one per grade) and 1 row
# figsize=(20, 5) gives each image roughly 4 inches of width
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

# Loop through each KL grade
for ax, grade in zip(axes, KL_GRADES):

    # Get path to this grade's training folder
    grade_folder = TRAIN_DIR / grade

    # Get list of all image files in this folder
    image_files = os.listdir(grade_folder)

    # Pick the first image (index 0)
    # random.choice() would pick randomly — we use index 0 for reproducibility
    chosen_image = image_files[0]

    # Build the full path to that image file
    image_path = grade_folder / chosen_image

    # Open the image using PIL
    # PIL opens it as a PIL Image object, not a numpy array yet
    img = Image.open(image_path)

    # Convert to grayscale to see the X-ray clearly
    # 'L' means luminance (grayscale) in PIL
    img_gray = img.convert('L')

    # Display the image on this subplot
    # cmap='gray' tells matplotlib to display it in greyscale
    ax.imshow(img_gray, cmap='gray')

    # Set the title above each image showing the grade and clinical name
    ax.set_title(GRADE_NAMES[grade], fontsize=11, fontweight='bold', pad=8)

    # Remove the x and y axis tick marks — they don't mean anything for images
    ax.axis('off')

# Main title
fig.suptitle(
    'Example Knee Radiograph per Kellgren-Lawrence Grade (Training Set)',
    fontsize=13, fontweight='bold', y=1.02
)

plt.tight_layout()

# Save as the figure referenced in your report (phase1_example_row.png)
save_path = f"{PROJECT}/outputs/figures/phase1_example_row.png"
plt.savefig(save_path, bbox_inches='tight', dpi=150)
print(f'Figure 1 saved to: {save_path}')

plt.show()

## CELL 8 — Image Size Audit
Check whether all images have the same dimensions.
Inconsistent sizes can cause errors in the DataLoader and need to be handled in preprocessing.

In [ ]:
# We will collect image sizes from a sample of 20 images per grade
# Checking all ~9,786 would take too long — a sample is sufficient
SAMPLE_SIZE = 20  # number of images to check per grade

print('Image size audit (sample of 20 images per grade from training set):')
print(f'{"Grade":<10} {"Width range":<20} {"Height range":<20} {"Consistent?"}')
print('-' * 65)

all_sizes = []  # store all sizes for overall summary

for grade in KL_GRADES:
    grade_folder = TRAIN_DIR / grade
    image_files = os.listdir(grade_folder)

    # Take a random sample to avoid bias from just checking the first images
    sample = random.sample(image_files, min(SAMPLE_SIZE, len(image_files)))

    # Collect width and height for each sampled image
    widths, heights = [], []
    for fname in sample:
        img = Image.open(grade_folder / fname)
        w, h = img.size  # PIL .size returns (width, height) as a tuple
        widths.append(w)
        heights.append(h)
        all_sizes.append((w, h))

    # Check if all images in this grade have the same size
    consistent = len(set(zip(widths, heights))) == 1  # set removes duplicates

    print(f'KL{grade:<9} '
          f'{min(widths)}-{max(widths):>6}px    '
          f'{min(heights)}-{max(heights):>6}px    '
          f'{"Yes" if consistent else "No — resizing needed"}')

print()
# Overall summary
all_widths = [s[0] for s in all_sizes]
all_heights = [s[1] for s in all_sizes]
print(f'Overall width range:  {min(all_widths)}px to {max(all_widths)}px')
print(f'Overall height range: {min(all_heights)}px to {max(all_heights)}px')
print()
print('Note: All images will be resized to 224x224 in preprocessing.')
print('This matches ResNet50 and EfficientNet-B0 input requirements.')

## CELL 9 — Pixel Intensity Analysis
Understand the brightness and contrast of images per grade.
This informs whether CLAHE preprocessing is likely to help.

**What to look for:**
- If KL0 and KL1 have very similar intensity distributions, it explains why the model confuses them
- If contrast is low across all grades, CLAHE is likely to help

In [ ]:
# For each grade, calculate mean pixel intensity and standard deviation
# across a sample of images
SAMPLE_SIZE = 30  # slightly larger sample for intensity analysis

intensity_stats = {}  # dictionary to store results

for grade in KL_GRADES:
    grade_folder = TRAIN_DIR / grade
    image_files = os.listdir(grade_folder)
    sample = random.sample(image_files, min(SAMPLE_SIZE, len(image_files)))

    means, stds = [], []

    for fname in sample:
        img = Image.open(grade_folder / fname).convert('L')  # grayscale

        # Convert PIL image to numpy array for numerical operations
        # Pixel values range from 0 (black) to 255 (white) in uint8 format
        img_array = np.array(img)

        # Calculate mean and standard deviation of pixel values
        # Mean = average brightness of the image
        # Std = how much contrast the image has (higher std = more contrast)
        means.append(img_array.mean())
        stds.append(img_array.std())

    intensity_stats[grade] = {
        'mean_brightness': np.mean(means),
        'std_brightness': np.std(means),
        'mean_contrast': np.mean(stds)
    }

# Plot intensity distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

grades = [f'KL{g}' for g in KL_GRADES]
mean_brightness = [intensity_stats[g]['mean_brightness'] for g in KL_GRADES]
mean_contrast   = [intensity_stats[g]['mean_contrast'] for g in KL_GRADES]

# Left plot — mean brightness per grade
axes[0].bar(grades, mean_brightness, color=GRADE_COLORS, edgecolor='white')
axes[0].set_title('Mean Pixel Brightness per KL Grade', fontweight='bold')
axes[0].set_ylabel('Mean pixel value (0=black, 255=white)')
axes[0].set_ylim(0, 255)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Right plot — mean contrast (std) per grade
axes[1].bar(grades, mean_contrast, color=GRADE_COLORS, edgecolor='white')
axes[1].set_title('Mean Image Contrast (Std Dev) per KL Grade', fontweight='bold')
axes[1].set_ylabel('Pixel standard deviation')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()

save_path = f"{PROJECT}/outputs/figures/intensity_analysis.png"
plt.savefig(save_path, bbox_inches='tight', dpi=150)
print(f'Intensity figure saved to: {save_path}')

plt.show()

# Print table summary
print('\nIntensity statistics per grade:')
for grade in KL_GRADES:
    s = intensity_stats[grade]
    print(f"  KL{grade}: brightness={s['mean_brightness']:.1f}, "
          f"contrast(std)={s['mean_contrast']:.1f}")

## CELL 10 — EDA Summary for Your Report
This cell prints a clean summary of everything this notebook found.
Copy these numbers directly into your Methods — Dataset section.

In [ ]:
print('=' * 65)
print('EDA SUMMARY — copy these numbers into your report')
print('=' * 65)

print()
print('DATASET SIZE')
total_train = sum(counts['train'].values())
total_val   = sum(counts['val'].values())
total_test  = sum(counts['test'].values())
total_all   = total_train + total_val + total_test
print(f'  Total images: {total_all}')
print(f'  Train: {total_train} | Val: {total_val} | Test: {total_test}')

print()
print('CLASS DISTRIBUTION (training set)')
for grade in KL_GRADES:
    n = counts['train'][grade]
    pct = n / total_train * 100
    print(f'  KL{grade} ({GRADE_NAMES[grade].split("—")[1].strip():12s}): '
          f'n={n:5d} ({pct:.1f}%)')

print()
print('IMBALANCE')
train_counts_list = [counts['train'][g] for g in KL_GRADES]
print(f'  Max class: {max(train_counts_list)} images (KL{KL_GRADES[train_counts_list.index(max(train_counts_list))]})')
print(f'  Min class: {min(train_counts_list)} images (KL{KL_GRADES[train_counts_list.index(min(train_counts_list))]})')
print(f'  Imbalance ratio: {max(train_counts_list)/min(train_counts_list):.1f}:1')

print()
print('CLASS WEIGHTS FOR LOSS FUNCTION')
train_arr = np.array(train_counts_list, dtype=np.float32)
weights = total_train / (5 * train_arr)
weights = weights / weights.sum() * 5
for grade, w in zip(KL_GRADES, weights):
    print(f'  KL{grade}: {w:.4f}')

print()
print('FIGURES SAVED')
print(f'  outputs/figures/class_distribution.png')
print(f'  outputs/figures/phase1_example_row.png  <-- Figure 1 in report')
print(f'  outputs/figures/intensity_analysis.png')

print()
print('NEXT STEP')
print('  Open 02_baseline_debug.ipynb')
print('  Test the DataLoader and model forward pass before HPC training')
print('=' * 65)